In [1]:
!nvidia-smi

Thu Oct  8 07:05:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
from numba import cuda
import os
device = cuda.get_current_device()
print(f"Device Name: {device.name}")
print(f"Compute Capability: {device.compute_capability}")
print(f"Multiprocessor Count (SMs): {getattr(device, 'MULTIPROCESSOR_COUNT', 'N/A')}")
print(f"Max Threads Per Block: {device.MAX_THREADS_PER_BLOCK}")
print(f"Max Block Dimensions: {device.MAX_BLOCK_DIM_X}, {device.MAX_BLOCK_DIM_Y}, {device.MAX_BLOCK_DIM_Z}")
print(f"Warp Size: {device.WARP_SIZE}")

Device Name: Tesla T4
Compute Capability: ComputeCapability(major=7, minor=5)
Multiprocessor Count (SMs): 40
Max Threads Per Block: 1024
Max Block Dimensions: 1024, 1024, 64
Warp Size: 32


In [13]:
import numpy as np
import time
import torch


from numba import cuda

# 1. CPU Reference Implementation (Pure Python loop)
def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out
# 2. GPU Kernel Implementation
@cuda.jit
def double_gpu_kernel(d_in, d_out):
    # Calculate unique 1D thread ID
    idx = cuda.grid(1)
    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))

GPU available: True
GPU name: Tesla T4


In [14]:
import torch
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Create some random data on CPU
x_cpu = torch.randn(10000, 10000)

# Measure roundtrip time: CPU → GPU → CPU
start = time.perf_counter()

# Transfer to GPU
x_gpu = x_cpu.to(device)

# Do a simple GPU computation
y_gpu = torch.matmul(x_gpu, x_gpu)

# Transfer result back to CPU
y_cpu = y_gpu.to("cpu")

end = time.perf_counter()

print("Total Roundtrip GPU Time:", end - start, "seconds")


Using device: cuda
Total Roundtrip GPU Time: 1.01010188500004 seconds


In [15]:
import numpy as np
import time
from numba import cuda

# 1. CPU Reference Implementation
def double_cpu_pure(arr):
    out = np.empty_like(arr)

    for i in range(len(arr)):
        out[i] = arr[i] * 2.0

    return out


# 2. GPU Kernel Implementation
@cuda.jit
def double_gpu_kernel(d_in, d_out):
    idx = cuda.grid(1)

    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0

In [24]:
def benchmark_run(N, student_id_seed=100):
    np.random.seed(student_id_seed)
    h_in = np.random.rand(N).astype(np.float32)

    # Measure CPU Pure Loop (skip for N >= 10M if too slow)
    if N <= 1_000_000:
        t0 = time.perf_counter()
        _ = double_cpu_pure(h_in)
        cpu_time_ms = (time.perf_counter() - t0) * 1000.0
    else:
        cpu_time_ms = float('nan') # Extrapolate or skip

    # GPU Setup
    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    # Warmup compilation run
    d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](d_warmup, d_warmup)
    cuda.synchronize()

    # GPU Roundtrip Measurement (Includes PCIe transfers)
    t0 = time.perf_counter()
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    h_out = d_out.copy_to_host()
    cuda.synchronize()
    gpu_total_ms = (time.perf_counter() - t0) * 1000.0

    # GPU Compute-Only Measurement (Pre-allocated in VRAM)
    t0 = time.perf_counter()
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    cuda.synchronize()
    gpu_compute_ms = (time.perf_counter() - t0) * 1000.0

    return cpu_time_ms, gpu_total_ms, gpu_compute_ms
sizes = [100, 1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]
rows = []
print(f"{'N':>12} | {'CPU ms':>10} | {'GPU total':>10} | {'GPU kernel':>11} | Winner")
for N in sizes:
    runs = np.array([benchmark_run(N) for _ in range(3)])
    cpu, tot, ker = np.median(runs, axis=0)
    winner = "GPU" if tot < cpu else "CPU"
    rows.append((N, cpu, tot, ker, winner))
    print(f"{N:>12,} | {cpu:>10.4f} | {tot:>10.4f} | {ker:>11.4f} | {winner}")

# crossover: first N from which GPU total stays faster than CPU
CROSSOVER_N = None
for i, r in enumerate(rows):
    if all(x[4] == "GPU" for x in rows[i:]):
        CROSSOVER_N = r[0]; break
print("Crossover N* =", CROSSOVER_N)

           N |     CPU ms |  GPU total |  GPU kernel | Winner


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


         100 |     0.0339 |     0.4227 |      0.0624 | CPU
       1,000 |     0.2694 |     0.4248 |      0.0615 | CPU
      10,000 |     2.2881 |     0.4102 |      0.0556 | GPU
     100,000 |    23.4498 |     0.6695 |      0.0639 | GPU
   1,000,000 |   243.2075 |     4.5146 |      0.1162 | GPU
   5,000,000 |        nan |    13.8256 |      0.2979 | CPU
  10,000,000 |        nan |    25.6801 |      0.4917 | CPU
Crossover N* = None


Q1 answer template: "On my T4 the crossover point is N* = none (at N = 100 GPU total was 0.4227 ms vs CPU 0.033 ms, and the GPU stayed faster for all larger N)."

Q2 answer template:
"At N = 100 the CPU needs only 0.033 ms, while GPU Total is 0.422 ms, but GPU Kernel Only is just 0.062 ms. So (total − kernel)/total = 14% of the time is not computation. It is fixed overhead: allocating device memory, the Host→Device and Device→Host copies over PCIe (each has a fixed latency of several microseconds no matter how few bytes move), the driver and Numba launch overhead, and synchronization. 100 floats is 400 bytes, so the transfer is pure latency, and the kernel does about 100 multiplications in microseconds. The GPU's parallelism cannot pay for this fixed cost until N is large enough."

In [25]:
# DEFECTIVE KERNEL: Notice the missing 'if idx < n' check!
@cuda.jit
def defective_kernel(d_arr):
    idx = cuda.grid(1)
    # FORBIDDEN: Writing without bounds check!
    d_arr[idx] = 999.0
# TEST HARNESS
N = 1000
h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)
# Launch 4 blocks of 256 = 1024 threads
threads_per_block = 256
blocks_per_grid = 4
print("Launching defective kernel with 1024 threads on 1000-element array...")
defective_kernel[blocks_per_grid, threads_per_block](d_arr)
cuda.synchronize()
print("Execution finished (or did it crash?). Checking host copy...")
h_result = d_arr.copy_to_host()
print(f"Elements processed: {len(h_result)}. Last element: {h_result[-1]}")

Launching defective kernel with 1024 threads on 1000-element array...
Execution finished (or did it crash?). Checking host copy...
Elements processed: 1000. Last element: 999.0


In [27]:
import numpy as np
from numba import cuda

@cuda.jit
def inspect_threads_kernel(log_block, log_thread, log_global, log_active, N):
    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)
    if g_idx < log_block.shape[0]:
        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0

# Configuration
N = 37
threads_per_block = 8
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block  # Exactly 5 blocks = 40 threads

log_b = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_t = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_g = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_a = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)

inspect_threads_kernel[blocks_per_grid, threads_per_block](log_b, log_t, log_g, log_a, N)
cuda.synchronize()

# copy to host once (faster than indexing device arrays one by one)
h_b, h_t, h_g, h_a = log_b.copy_to_host(), log_t.copy_to_host(), log_g.copy_to_host(), log_a.copy_to_host()

print(f"{'Global ID':<10} | {'Block ID':<10} | {'Thread ID':<10} | {'Status':<10}")
print("-" * 50)
for i in range(32, 40):  # critical boundary transition (Block 4)
    status = "ACTIVE" if h_a[i] == 1 else "IDLE (GUARDED)"
    print(f"{h_g[i]:<10} | {h_b[i]:<10} | {h_t[i]:<10} | {status:<10}")

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Global ID  | Block ID   | Thread ID  | Status    
--------------------------------------------------
32         | 4          | 0          | ACTIVE    
33         | 4          | 1          | ACTIVE    
34         | 4          | 2          | ACTIVE    
35         | 4          | 3          | ACTIVE    
36         | 4          | 4          | ACTIVE    
37         | 4          | 5          | IDLE (GUARDED)
38         | 4          | 6          | IDLE (GUARDED)
39         | 4          | 7          | IDLE (GUARDED)


In [29]:
import numpy as np
import time
from numba import cuda

# BUGGY NAIVE SUMMATION
@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        d_total[0] += d_arr[idx]   # data race

# CORRECT ATOMIC SUMMATION
@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        cuda.atomic.add(d_total, 0, d_arr[idx])

N = 50_000
d_ones = cuda.to_device(np.ones(N, dtype=np.float32))
tpb = 256
bpg = (N + tpb - 1) // tpb

# warm-up so compile time is not measured
w = cuda.to_device(np.zeros(1, dtype=np.float32))
naive_sum_kernel[bpg, tpb](d_ones, w)
atomic_sum_kernel[bpg, tpb](d_ones, w)
cuda.synchronize()

print("Trial | Output | Lost updates")
for t in range(1, 6):
    d_total = cuda.to_device(np.zeros(1, dtype=np.float32))   # reset each trial
    naive_sum_kernel[bpg, tpb](d_ones, d_total)
    cuda.synchronize()
    v = d_total.copy_to_host()[0]
    print(f"{t} | {v} | {50000.0 - v}")

# atomic result
d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
atomic_sum_kernel[bpg, tpb](d_ones, d_total)
cuda.synchronize()
print("Atomic:", d_total.copy_to_host()[0])

# timing (average of 50 launches each)
def time_kernel(k, reps=50):
    t0 = time.perf_counter()
    for _ in range(reps):
        d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
        k[bpg, tpb](d_ones, d_total)
        cuda.synchronize()
    return (time.perf_counter() - t0) / reps * 1000

print(f"Naive: {time_kernel(naive_sum_kernel):.4f} ms | Atomic: {time_kernel(atomic_sum_kernel):.4f} ms")

Trial | Output | Lost updates
1 | 2.0 | 49998.0
2 | 2.0 | 49998.0
3 | 2.0 | 49998.0
4 | 2.0 | 49998.0
5 | 2.0 | 49998.0
Atomic: 50000.0
Naive: 0.2774 ms | Atomic: 0.4527 ms


Calculation answers:

Total blocks: 5
Total threads launched: 40
Active threads: 37
Idle guarded threads: 3 (global IDs 37, 38, 39)

In [33]:
import math
@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):
    # Retrieve 2D spatial coordinates directly from the grid
    x, y = cuda.grid(2)

    if x < width and y < height:
        dx = float(x - center_x)
        dy = float(y - center_y)
        dist = math.sqrt(dx * dx + dy * dy)
        max_dist = math.sqrt(float(center_x**2 + center_y**2))
        intensity = 1.0 - (dist / max_dist)
        if intensity < 0.0:
            intensity = 0.0
        d_canvas[y, x] = intensity
# 2D EXECUTION CONFIGURATION
WIDTH, HEIGHT = 1024, 1024
canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)
# 2D Blocks (16x16 = 256 threads per block)
threads_2d = (16, 16)
blocks_2d = (
    (WIDTH + threads_2d[0] - 1) // threads_2d[0],
    (HEIGHT + threads_2d[1] - 1) // threads_2d[1]
)
radial_vignette_2d[blocks_2d, threads_2d](d_canvas, WIDTH, HEIGHT, WIDTH // 2, HEIGHT // 2)
cuda.synchronize()
result_canvas = d_canvas.copy_to_host()
print(f"Generated 2D matrix shape: {result_canvas.shape}. Center value: {result_canvas[512, 512]:.3f}")
print(result_canvas[0, 0])

Generated 2D matrix shape: (1024, 1024). Center value: 1.000
0.0


In [35]:
import hashlib
def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()
    h.update(student_id_str.strip().encode('utf-8'))
    gpu_name = cuda.get_current_device().name
    h.update(gpu_name if isinstance(gpu_name, bytes) else gpu_name.encode('utf-8'))
    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())
    h.update(str(exp_table_crossover).encode('utf-8'))
    digest = h.hexdigest()[:16].upper()
    print("=" * 60); print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}"); print("=" * 60)
    return digest

STUDENT_ID = "230103112"
generate_lab_checksum(STUDENT_ID, d_canvas, CROSSOVER_N)

OFFICIAL VERIFICATION CHECKSUM TOKEN: CE2937A85057F8CC


'CE2937A85057F8CC'